# Nemotron LoRA — OFFLINE train on the air-gapped Blackwell box

For an **RTX PRO 6000 Blackwell (96 GB, sm_120)** with **no internet**. Inputs come
from two attached datasets staged by `offline_staging.ipynb`:

- **offline bundle** — `wheels/`, `repo/`, `data/train_sft.jsonl`
- **model** — the ~63 GB `NVIDIA-Nemotron-3-Nano-30B-A3B-BF16` folder

Strategy: **bf16, no quantization** (96 GB fits the full model — no bitsandbytes, no
4-bit, so the lm_head grad bug can't occur), `adamw_torch` optimizer, fast Mamba
kernel with a `torch_forward` fallback. Full epoch ≈ 2 h.


## 0. Hardware check (expect sm_120 / ~96 GB)

In [ ]:
import torch
print('GPUs:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'  [{i}] {p.name}  {p.total_memory/1e9:.0f} GB  sm_{p.major}{p.minor}')
print('torch', torch.__version__, 'cuda', torch.version.cuda)


## 1. Locate the attached datasets
Edit these two paths to wherever your platform mounts the datasets (the defaults
glob common mount roots). The bundle has `wheels/ repo/ data/`; the model folder has
`config.json` + `*.safetensors`.

In [ ]:
import os, glob, shutil
def find(patterns):
    for p in patterns:
        h = glob.glob(p, recursive=True)
        if h: return sorted(h)[0]
    return None

# bundle root = the folder that CONTAINS wheels/ and repo/
bundle = find(['/kaggle/input/**/offline_bundle', '/mnt/**/offline_bundle',
               '/data/**/offline_bundle', '/workspace/**/offline_bundle'])
if bundle is None:
    w = find(['/**/offline_bundle/wheels']);  bundle = os.path.dirname(w) if w else None
assert bundle, 'Could not find offline_bundle — set `bundle = "/your/mount/offline_bundle"` manually.'

# model root = the folder that CONTAINS config.json + safetensors
mcfg = find(['/kaggle/input/**/config.json', '/mnt/**/config.json',
             '/data/**/config.json', '/workspace/**/config.json'])
MODEL = os.path.dirname(mcfg) if mcfg else None
assert MODEL and glob.glob(MODEL+'/*.safetensors'), 'Could not find the model folder — set MODEL manually.'
print('bundle:', bundle)
print('model :', MODEL)


## 2. Offline install from the wheel bundle (no network)

In [ ]:
WH = os.path.join(bundle, 'wheels')
# everything except mamba/causal (uses the box's already-installed torch 2.10+cu128)
!pip install -q --no-index --find-links {WH} \
    'transformers>=4.45,<5' peft trl datasets accelerate einops sentencepiece psutil safetensors huggingface_hub tokenizers
# mamba + causal_conv1d, no deps (must IMPORT even if we fall back to torch_forward)
!pip install -q --no-index --no-deps {WH}/causal_conv1d-*.whl {WH}/mamba_ssm-*.whl
import importlib
for m in ('transformers','peft','trl','accelerate','mamba_ssm'):
    importlib.import_module(m); print('ok', m)


## 3. Stage repo + data into a writable workspace

In [ ]:
WORK = '/tmp/nemotron'
!rm -rf {WORK} && mkdir -p {WORK}
!cp -r {bundle}/repo {WORK}/repo
!mkdir -p {WORK}/repo/data && cp {bundle}/data/train_sft.jsonl {WORK}/repo/data/
os.chdir(f'{WORK}/repo')
print('cwd:', os.getcwd(), '| sft rows:', sum(1 for _ in open('data/train_sft.jsonl')))


## 4. Train — bf16, no quant, on the 96 GB card
First try the fast fused Mamba kernel. If it errors on sm_120 (Blackwell), set
`FORCE_TORCH_FORWARD=1` in the env below and re-run — on this card the fallback is
still fast and won't OOM. The adapter saves every 100 steps to `OUT`.

In [ ]:
import os
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['MODEL_PATH'] = MODEL
os.environ['QUANT'] = 'none'                      # full bf16 (fits 96 GB)
os.environ['OPTIM'] = 'adamw_torch'               # no bitsandbytes dependency
os.environ['NEMOTRON_MAX_MEMORY_GPU'] = '90GiB'   # keep it all on-GPU
os.environ['SFT_MAX_SEQ_LENGTH'] = '1024'
os.environ['NUM_EPOCHS'] = '1'
os.environ['SAVE_STEPS'] = '100'
# os.environ['FORCE_TORCH_FORWARD'] = '1'         # <- uncomment if the fused kernel errors on sm_120

OUT = f'{WORK}/lora_adapter'
!python scripts/03_train_lora.py --data-path data/train_sft.jsonl --output-dir {OUT}


## 5. Package -> submission.zip (rank 16 ≤ 32)

In [ ]:
!python scripts/05_package_submission.py --adapter-dir {OUT} --output {WORK}/submission.zip
!ls -lh {WORK}/submission.zip
print('Copy', WORK+'/submission.zip', 'off the box (or write it to an output dataset) and submit.')
